# 08a — AWS Braket SV1 QAOA (Scenario B)

This notebook runs **Scenario B** on **AWS Braket SV1** using a simple **QAOA p=1** grid search over angles (γ, β).

Key points:
- We load the Scenario B QUBO created in `03b` (`data/qubo_scenarios/scenario_B_qubo.json`).
- We submit Braket SV1 quantum tasks and store *task outputs* in the required **amazon-braket-*** bucket.
- We export clean “portfolio artifacts” locally and upload them to the **project S3 bucket** for your showcase.

Outputs (local + pushed to GitHub):
- `data/results/scenario_B_braket_sv1_summary.csv`
- `data/results/scenario_B_braket_sv1_selected_trials.csv`
- `data/results/scenario_B_braket_sv1_job_metadata.json`
- `data/results/scenario_B_braket_input_bundle.json`

Uploads (project S3 bucket):
- `s3://quantum-clinical-optimization-us-west-2/<prefix>/scenario_B_braket_*`


In [1]:
# ============================================================
# Cell 1 — Setup: imports, paths, and required artifacts
# ============================================================

from pathlib import Path
import json
import time
import numpy as np
import pandas as pd

# --- Local paths ---
DATA_DIR = Path("data")
QUBO_DIR = DATA_DIR / "qubo_scenarios"
RESULTS_DIR = DATA_DIR / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

PATH_QUBO = QUBO_DIR / "scenario_B_qubo.json"
if not PATH_QUBO.exists():
    raise FileNotFoundError(f"Missing: {PATH_QUBO}. Run 03b first.")

# --- Outputs ---
PATH_BRK_SUMMARY  = RESULTS_DIR / "scenario_B_braket_sv1_summary.csv"
PATH_BRK_SELECTED = RESULTS_DIR / "scenario_B_braket_sv1_selected_trials.csv"
PATH_BRK_JOB_META = RESULTS_DIR / "scenario_B_braket_sv1_job_metadata.json"
PATH_INPUT_BUNDLE = RESULTS_DIR / "scenario_B_braket_input_bundle.json"

print("OK: Found Scenario B QUBO:", PATH_QUBO)
print("Will write:")
print("  -", PATH_BRK_SUMMARY)
print("  -", PATH_BRK_SELECTED)
print("  -", PATH_BRK_JOB_META)
print("  -", PATH_INPUT_BUNDLE)


OK: Found Scenario B QUBO: data/qubo_scenarios/scenario_B_qubo.json
Will write:
  - data/results/scenario_B_braket_sv1_summary.csv
  - data/results/scenario_B_braket_sv1_selected_trials.csv
  - data/results/scenario_B_braket_sv1_job_metadata.json
  - data/results/scenario_B_braket_input_bundle.json


### What Cell 1 Just Did

- Verified the Scenario B QUBO JSON exists (`03b` output).
- Declared the canonical local artifact paths for the Braket SV1 run.


In [2]:
# ============================================================
# Cell 2 — AWS + Braket session setup (SV1 device + S3 destinations) as cell 4 was stuck and interupted in the terminal
# ============================================================

import os
import boto3
from braket.aws import AwsSession, AwsDevice

# --- Region (match your project bucket region) ---
AWS_REGION = os.environ.get("AWS_REGION", "us-west-2")

# --- Your project bucket (for clean showcase artifacts) ---
PROJECT_BUCKET = "quantum-clinical-optimization-us-west-2"
PROJECT_PREFIX_RESULTS = "clinical-trials-data/results"   # adjust if you prefer
PROJECT_PREFIX_QUBO = "clinical-trials-data/qubo_scenarios"

# --- Braket SV1 device ARN ---
SV1_ARN = "arn:aws:braket:::device/quantum-simulator/amazon/sv1"

# Build a boto3 Session (region-aware)
boto_sess = boto3.session.Session(region_name=AWS_REGION)
s3 = boto_sess.client("s3")

# Braket session must be an AwsSession (not a raw boto3 Session)
braket_sess = AwsSession(boto_session=boto_sess)

# Braket task results MUST go to an amazon-braket-* bucket (Braket default bucket is safest)
BRAKET_BUCKET = braket_sess.default_bucket()
BRAKET_TASK_PREFIX = f"{PROJECT_PREFIX_RESULTS}/sv1_tasks/scenario_B"

device = AwsDevice(SV1_ARN, aws_session=braket_sess)

print("AWS region:", AWS_REGION)
print("Braket device:", device.name)
print("Braket default bucket (task outputs):", BRAKET_BUCKET)
print("Braket task prefix:", f"s3://{BRAKET_BUCKET}/{BRAKET_TASK_PREFIX}/")
print("Project bucket (showcase artifacts):", f"s3://{PROJECT_BUCKET}/{PROJECT_PREFIX_RESULTS}/")


AWS region: us-west-2
Braket device: SV1
Braket default bucket (task outputs): amazon-braket-us-west-2-581610642254
Braket task prefix: s3://amazon-braket-us-west-2-581610642254/clinical-trials-data/results/sv1_tasks/scenario_B/
Project bucket (showcase artifacts): s3://quantum-clinical-optimization-us-west-2/clinical-trials-data/results/


### What Cell 2 Just Did

- Created a proper Braket `AwsSession` (avoids the common `.region` AttributeError from using a raw boto3 Session incorrectly).
- Connected to the SV1 simulator device.
- Configured **task outputs** to write to the required `amazon-braket-*` bucket while keeping your “showcase artifacts” in your project bucket.


In [3]:
# ============================================================
# Cell 3 — Load QUBO + convert to Ising + define decoding utilities
# ============================================================

from braket.circuits import Circuit

with open(PATH_QUBO, "r") as f:
    payload = json.load(f)

if not isinstance(payload, dict) or "Q" not in payload:
    raise ValueError(f"scenario_B_qubo.json missing key 'Q'. Keys: {list(payload.keys()) if isinstance(payload, dict) else type(payload)}")

Q = np.array(payload["Q"], dtype=float)
n = int(Q.shape[0])

nct_ids = payload.get("nct_ids", None)
if not nct_ids or len(nct_ids) != n:
    raise ValueError("scenario_B_qubo.json must include nct_ids aligned to Q dimension.")

def qubo_energy(Q, x01):
    x = np.asarray(x01, dtype=float).reshape(-1, 1)
    return float((x.T @ Q @ x)[0, 0])

def bitstring_to_x_lr(s, n):
    """Interpret s left->right as x[0],x[1],..."""
    s = str(s)
    if len(s) != n:
        s = s[:n].ljust(n, "0")
    return np.array([1 if ch == "1" else 0 for ch in s], dtype=int)

def bitstring_to_x_rl(s, n):
    """Interpret s right->left as x[0],x[1],... (reverse)"""
    s = str(s)
    if len(s) != n:
        s = s[:n].ljust(n, "0")
    s = s[::-1]
    return np.array([1 if ch == "1" else 0 for ch in s], dtype=int)

def best_energy_for_bitstring(s, Q):
    """Braket bit ordering can differ; evaluate both orientations and keep the better one."""
    n = Q.shape[0]
    x_lr = bitstring_to_x_lr(s, n)
    x_rl = bitstring_to_x_rl(s, n)
    E_lr = qubo_energy(Q, x_lr)
    E_rl = qubo_energy(Q, x_rl)
    if E_lr <= E_rl:
        return E_lr, "LR", x_lr
    return E_rl, "RL", x_rl

def qubo_to_ising(Q):
    """
    Convert QUBO (x in {0,1}) to Ising (s in {-1,+1}) for cost Hamiltonian:
      x = (1 - s)/2
    Returns h (local fields) and J (couplers) for an energy:
      E(s) = sum_i h_i s_i + sum_{i<j} J_ij s_i s_j + const
    """
    Q = np.array(Q, dtype=float)
    n = Q.shape[0]
    Qsym = 0.5 * (Q + Q.T)

    h = np.zeros(n, dtype=float)
    J = {}

    # Expand x^T Q x with x=(1-s)/2; keep only terms needed for QAOA phase separation.
    # This form works well for p=1 demonstrations.
    for i in range(n):
        for j in range(n):
            q = Qsym[i, j]
            if q == 0.0:
                continue
            # Contributions:
            # x_i x_j = (1 - s_i - s_j + s_i s_j)/4
            # Add to h and J
            h[i] += (-q) / 4.0
            h[j] += (-q) / 4.0
            if i != j:
                a, b = (i, j) if i < j else (j, i)
                J[(a, b)] = J.get((a, b), 0.0) + (q) / 4.0
            else:
                # diagonal term also contributes constant + s_i terms; handled via h update above
                pass

    # Correct for double-counting because we iterated all i,j
    h *= 0.5
    for k in list(J.keys()):
        J[k] *= 0.5

    return h, J

h, J = qubo_to_ising(Q)
print("Loaded Scenario B QUBO.")
print("n:", n, "| #couplers:", len(J), "| max|h|:", float(np.max(np.abs(h))))


Loaded Scenario B QUBO.
n: 60 | #couplers: 1770 | max|h|: 100.25


### What Cell 3 Just Did

- Loaded Scenario B’s QUBO matrix and its deterministic `nct_ids` mapping.
- Converted the QUBO into Ising-style coefficients used to build the QAOA cost unitary.
- Implemented robust decoding that evaluates **both possible bitstring orientations** and picks the one with lower QUBO energy (prevents “bit order” surprises).


In [6]:
# ============================================================
# Cell 4 — SV1-compatible QAOA in a separate process (pool <=34 qubits + small grid)
# ============================================================

import subprocess, textwrap, json, os, sys
from pathlib import Path

# --- SV1 limits + pooling ---
MAX_QUBITS = 34
POOL_STRATEGY = "best_diag"   # deterministic, good baseline under minimization

# --- Demo mode (FAST, recommended first run) ---
SHOTS = 50
GAMMA_GRID = [0.6, 1.0]
BETA_GRID  = [0.6, 1.0]

# --- Portfolio-grade mode (uncomment for a nicer sweep) ---
# SHOTS = 100
# GAMMA_GRID = [0.4, 0.9, 1.4]
# BETA_GRID  = [0.4, 0.9, 1.4]

# --- Braket results destination (must be amazon-braket-* bucket) ---
s3_destination_folder = (BRAKET_BUCKET, BRAKET_TASK_PREFIX)

# --- Where the child process will write the best result ---
BEST_JSON = RESULTS_DIR / "_08a_sv1_best_scenario_B.json"
RUNNER = RESULTS_DIR / "_run_08a_sv1_scenario_B.py"

runner_code = f"""
import os, json, time
import numpy as np
import boto3
from braket.aws import AwsSession, AwsDevice
from braket.circuits import Circuit

PATH_QUBO = r"{str(PATH_QUBO)}"
BEST_JSON = r"{str(BEST_JSON)}"

AWS_REGION = os.environ.get("AWS_REGION", "us-west-2")
SV1_ARN = "arn:aws:braket:::device/quantum-simulator/amazon/sv1"

BRAKET_BUCKET = os.environ.get("BRAKET_BUCKET")            # amazon-braket-* bucket
BRAKET_TASK_PREFIX = os.environ.get("BRAKET_TASK_PREFIX")  # prefix within bucket

MAX_QUBITS = int(os.environ.get("MAX_QUBITS", "34"))
POOL_STRATEGY = os.environ.get("POOL_STRATEGY", "best_diag")

SHOTS = int(os.environ.get("SHOTS", "50"))
GAMMA_GRID = json.loads(os.environ.get("GAMMA_GRID"))
BETA_GRID  = json.loads(os.environ.get("BETA_GRID"))

def qubo_energy(Q, x01):
    x = np.asarray(x01, dtype=float).reshape(-1, 1)
    return float((x.T @ Q @ x)[0, 0])

def bitstring_to_x_lr(s, n):
    s = str(s)
    if len(s) != n:
        s = s[:n].ljust(n, "0")
    return np.array([1 if ch == "1" else 0 for ch in s], dtype=int)

def bitstring_to_x_rl(s, n):
    s = str(s)
    if len(s) != n:
        s = s[:n].ljust(n, "0")
    s = s[::-1]
    return np.array([1 if ch == "1" else 0 for ch in s], dtype=int)

def best_energy_for_bitstring(s, Q):
    n = Q.shape[0]
    x_lr = bitstring_to_x_lr(s, n)
    x_rl = bitstring_to_x_rl(s, n)
    E_lr = qubo_energy(Q, x_lr)
    E_rl = qubo_energy(Q, x_rl)
    if E_lr <= E_rl:
        return E_lr, "LR", x_lr
    return E_rl, "RL", x_rl

def qubo_to_ising(Q):
    Q = np.array(Q, dtype=float)
    n = Q.shape[0]
    Qsym = 0.5 * (Q + Q.T)

    h = np.zeros(n, dtype=float)
    J = {{}}

    for i in range(n):
        for j in range(n):
            q = Qsym[i, j]
            if q == 0.0:
                continue
            h[i] += (-q) / 4.0
            h[j] += (-q) / 4.0
            if i != j:
                a, b = (i, j) if i < j else (j, i)
                J[(a, b)] = J.get((a, b), 0.0) + (q) / 4.0

    h *= 0.5
    for k in list(J.keys()):
        J[k] *= 0.5

    return h, J

def choose_pool_indices(Q_full, max_qubits=34, strategy="best_diag"):
    n_full = Q_full.shape[0]
    k = min(int(max_qubits), int(n_full))
    if strategy == "best_diag":
        diag = np.diag(Q_full)
        return np.argsort(diag)[:k].tolist()
    return list(range(k))

def submatrix(Q_full, idx):
    idx = np.asarray(idx, dtype=int)
    return Q_full[np.ix_(idx, idx)]

def qaoa_circuit_p1_braket(h, J, gamma, beta):
    n = len(h)
    c = Circuit()
    for q in range(n):
        c.h(q)
    for i, hi in enumerate(h):
        if hi != 0.0:
            c.rz(i, 2.0 * float(gamma) * float(hi))
    for (i, j), Jij in J.items():
        if Jij == 0.0:
            continue
        c.cnot(i, j)
        c.rz(j, 2.0 * float(gamma) * float(Jij))
        c.cnot(i, j)
    for q in range(n):
        c.rx(q, 2.0 * float(beta))
    for q in range(n):
        c.measure(q)
    return c

# --- Load QUBO ---
with open(PATH_QUBO, "r") as f:
    payload = json.load(f)

if not isinstance(payload, dict) or "Q" not in payload:
    raise ValueError("scenario_B_qubo.json must contain a top-level 'Q' key.")

Q = np.array(payload["Q"], dtype=float)
n_full = int(Q.shape[0])
nct_ids = payload.get("nct_ids", None)
if not nct_ids or len(nct_ids) != n_full:
    raise ValueError("scenario_B_qubo.json must include nct_ids aligned to Q dimension.")

# --- AWS/Braket session ---
boto_sess = boto3.session.Session(region_name=AWS_REGION)
braket_sess = AwsSession(boto_session=boto_sess)
device = AwsDevice(SV1_ARN, aws_session=braket_sess)

if not BRAKET_BUCKET or not BRAKET_TASK_PREFIX:
    # safest fallback: use session default bucket/prefix
    BRAKET_BUCKET = braket_sess.default_bucket()
    BRAKET_TASK_PREFIX = "tasks/scenario_B"

s3_destination_folder = (BRAKET_BUCKET, BRAKET_TASK_PREFIX)

# --- Pool to fit SV1 ---
pool_idx = choose_pool_indices(Q, max_qubits=MAX_QUBITS, strategy=POOL_STRATEGY)
Qsub = submatrix(Q, pool_idx)
h_sub, J_sub = qubo_to_ising(Qsub)

def run_counts_on_sv1(circuit, shots):
    task = device.run(circuit, shots=int(shots), s3_destination_folder=s3_destination_folder)
    result = task.result()
    return dict(result.measurement_counts), task

best = {{
    "gamma": None,
    "beta": None,
    "qubo_energy_full": float("inf"),
    "qubo_energy_sub": float("inf"),
    "bitstring": None,
    "bit_order": None,
    "task_arn": None,
    "counts_top": None,
    "pool_idx": pool_idx,
    "pool_strategy": POOL_STRATEGY,
    "n_full": n_full,
    "n_pool": len(pool_idx),
    "shots": SHOTS,
    "gamma_grid": GAMMA_GRID,
    "beta_grid": BETA_GRID,
    "braket_task_bucket": BRAKET_BUCKET,
    "braket_task_prefix": BRAKET_TASK_PREFIX,
}}

t0 = time.time()
print(f"SV1 run: n_full={{n_full}} -> n_pool={{len(pool_idx)}} (MAX_QUBITS={{MAX_QUBITS}}), grid={{len(GAMMA_GRID)}}x{{len(BETA_GRID)}}, shots={{SHOTS}}")
print(f"Task outputs: s3://{{BRAKET_BUCKET}}/{{BRAKET_TASK_PREFIX}}/")

for gamma in GAMMA_GRID:
    for beta in BETA_GRID:
        circ = qaoa_circuit_p1_braket(h_sub, J_sub, gamma=float(gamma), beta=float(beta))
        counts, task = run_counts_on_sv1(circ, SHOTS)

        local_best_Esub = float("inf")
        local_best_s = None
        local_best_order = None
        local_best_xsub = None

        for s, c in counts.items():
            Esub, order, xsub = best_energy_for_bitstring(s, Qsub)
            if Esub < local_best_Esub:
                local_best_Esub = Esub
                local_best_s = s
                local_best_order = order
                local_best_xsub = xsub

        x_full = np.zeros(n_full, dtype=int)
        for pos, orig_i in enumerate(pool_idx):
            x_full[orig_i] = int(local_best_xsub[pos])

        Efull = qubo_energy(Q, x_full)

        if float(Efull) < best["qubo_energy_full"]:
            best.update({{
                "gamma": float(gamma),
                "beta": float(beta),
                "qubo_energy_full": float(Efull),
                "qubo_energy_sub": float(local_best_Esub),
                "bitstring": str(local_best_s),
                "bit_order": str(local_best_order),
                "task_arn": getattr(task, "id", None) or getattr(task, "arn", None) or str(task),
                "counts_top": dict(sorted(counts.items(), key=lambda kv: kv[1], reverse=True)[:20]),
            }})

        print(f"gamma={{float(gamma):.3f}} beta={{float(beta):.3f}} | best_Esub={{local_best_Esub:.6f}} | best_Efull={{float(Efull):.6f}}")

with open(BEST_JSON, "w") as f:
    json.dump(best, f, indent=2)

print("Wrote:", BEST_JSON)
print("Best:", best)
print("Elapsed minutes:", (time.time() - t0)/60.0)
"""

RUNNER.write_text(textwrap.dedent(runner_code))
print("Wrote SV1 runner:", RUNNER)

# --- Run it as a subprocess (avoids Jupyter asyncio/contextvars issues) ---
env = os.environ.copy()
env["AWS_REGION"] = AWS_REGION
env["MAX_QUBITS"] = str(MAX_QUBITS)
env["POOL_STRATEGY"] = POOL_STRATEGY
env["SHOTS"] = str(SHOTS)
env["GAMMA_GRID"] = json.dumps(GAMMA_GRID)
env["BETA_GRID"] = json.dumps(BETA_GRID)

# Pass Braket bucket/prefix explicitly (must remain amazon-braket-* bucket)
env["BRAKET_BUCKET"] = BRAKET_BUCKET
env["BRAKET_TASK_PREFIX"] = BRAKET_TASK_PREFIX

cp = subprocess.run([sys.executable, str(RUNNER)], env=env, capture_output=True, text=True)
print(cp.stdout)
if cp.returncode != 0:
    print(cp.stderr)
    raise RuntimeError(f"SV1 runner failed with return code {cp.returncode}")

# --- Load best back into notebook state for Cell 5 ---
if not BEST_JSON.exists():
    raise FileNotFoundError(f"Runner did not write: {BEST_JSON}")

best = json.loads(BEST_JSON.read_text())
print("Loaded best back into notebook:")
print(best)


Wrote SV1 runner: data/results/_run_08a_sv1_scenario_B.py


KeyboardInterrupt: 

In [8]:
# ============================================================
# Cell 4b  — Recovery: rebuild best by reading Braket results.json from S3 (supports "measurements" format)
# ============================================================

import json
import numpy as np
from pathlib import Path
import boto3
from collections import Counter

RESULTS_DIR = Path("data/results")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

PATH_QUBO = Path("data/qubo_scenarios/scenario_B_qubo.json")
if not PATH_QUBO.exists():
    raise FileNotFoundError(f"Missing: {PATH_QUBO}")

AWS_REGION = "us-west-2"

# Braket output location (your account’s braket bucket + our prefix)
BRAKET_BUCKET = "amazon-braket-us-west-2-581610642254"
BRAKET_TASK_PREFIX = "clinical-trials-data/results/sv1_tasks/scenario_B"
BEST_JSON = RESULTS_DIR / "_08a_sv1_best_scenario_B.json"

# --- Load QUBO ---
payload = json.loads(PATH_QUBO.read_text())
if "Q" not in payload:
    raise ValueError("scenario_B_qubo.json must contain top-level key 'Q'.")

Q = np.array(payload["Q"], dtype=float)
n_full = int(Q.shape[0])
nct_ids = payload.get("nct_ids", [])
if len(nct_ids) != n_full:
    raise ValueError("scenario_B_qubo.json must include nct_ids aligned to Q dimension.")

def qubo_energy(Qm, x01):
    x = np.asarray(x01, dtype=float).reshape(-1, 1)
    return float((x.T @ Qm @ x)[0, 0])

# Same deterministic pooling as Cell 4 (SV1 <= 34 qubits)
MAX_QUBITS = 34
POOL_STRATEGY = "best_diag"
pool_idx = np.argsort(np.diag(Q))[:min(MAX_QUBITS, n_full)].tolist()
Qsub = Q[np.ix_(pool_idx, pool_idx)]

def bitstring_to_x_lr(s, n):
    s = str(s)
    if len(s) != n:
        s = s[:n].ljust(n, "0")
    return np.array([1 if ch == "1" else 0 for ch in s], dtype=int)

def bitstring_to_x_rl(s, n):
    s = str(s)
    if len(s) != n:
        s = s[:n].ljust(n, "0")
    s = s[::-1]
    return np.array([1 if ch == "1" else 0 for ch in s], dtype=int)

def best_energy_for_bitstring(s, Qm):
    """
    We score BOTH bit orders (LR and reversed) and take the best,
    so we don't care how Braket orders measured bits.
    """
    n = Qm.shape[0]
    x_lr = bitstring_to_x_lr(s, n)
    x_rl = bitstring_to_x_rl(s, n)
    E_lr = qubo_energy(Qm, x_lr)
    E_rl = qubo_energy(Qm, x_rl)
    if E_lr <= E_rl:
        return E_lr, "LR", x_lr
    return E_rl, "RL", x_rl

def extract_counts_from_results_json(obj):
    """
    Supports:
      - measurementCounts / measurement_counts style dict
      - "measurements" list (per-shot bit arrays)
    Returns: dict[str_bitstring] -> int
    """
    if isinstance(obj, dict):
        # Common direct-count forms
        for k in ("measurementCounts", "measurement_counts", "counts"):
            if k in obj and isinstance(obj[k], dict) and obj[k]:
                d = obj[k]
                # normalize spaces just in case
                return {str(bs).replace(" ", ""): int(ct) for bs, ct in d.items()}

        # Braket v1 results.json often has top-level "measurements"
        if "measurements" in obj and isinstance(obj["measurements"], list):
            meas = obj["measurements"]
            if len(meas) == 0:
                return {}

            # Each entry is usually a list of 0/1 bits. Sometimes it’s a string-like already.
            c = Counter()
            for row in meas:
                if isinstance(row, list) or isinstance(row, tuple) or isinstance(row, np.ndarray):
                    bs = "".join(str(int(b)) for b in row)
                else:
                    # fallback: try to stringify
                    bs = str(row).replace(" ", "")
                c[bs] += 1
            return dict(c)

    # Deep-search fallback (rare but safe)
    if isinstance(obj, dict):
        for v in obj.values():
            got = extract_counts_from_results_json(v)
            if got:
                return got
    elif isinstance(obj, list):
        for it in obj:
            got = extract_counts_from_results_json(it)
            if got:
                return got

    return None


# --- List results.json keys under the prefix ---
s3 = boto3.client("s3", region_name=AWS_REGION)

result_keys = []
paginator = s3.get_paginator("list_objects_v2")
for page in paginator.paginate(Bucket=BRAKET_BUCKET, Prefix=BRAKET_TASK_PREFIX + "/"):
    for obj in page.get("Contents", []):
        k = obj["Key"]
        if k.endswith("results.json"):
            result_keys.append(k)

result_keys = sorted(result_keys)
print(f"Found {len(result_keys)} results.json file(s) in s3://{BRAKET_BUCKET}/{BRAKET_TASK_PREFIX}/")
if not result_keys:
    raise FileNotFoundError("No results.json files found under the Braket prefix.")

best = {
    "source": "s3_results_json",
    "qubo_energy_full": float("inf"),
    "qubo_energy_sub": float("inf"),
    "bitstring": None,
    "bit_order": None,
    "counts_top": None,
    "results_json_key": None,
    "pool_idx": pool_idx,
    "pool_strategy": POOL_STRATEGY,
    "n_full": n_full,
    "n_pool": len(pool_idx),
    "braket_task_bucket": BRAKET_BUCKET,
    "braket_task_prefix": BRAKET_TASK_PREFIX,
}

for key in result_keys:
    raw = s3.get_object(Bucket=BRAKET_BUCKET, Key=key)["Body"].read().decode("utf-8")
    obj = json.loads(raw)

    counts = extract_counts_from_results_json(obj)
    if counts is None or len(counts) == 0:
        raise ValueError(
            f"Could not extract measurement counts from {key}. "
            f"Top-level keys: {list(obj.keys())[:20]}"
        )

    # Best in pooled space
    local_best_Esub = float("inf")
    local_best_s = None
    local_best_order = None
    local_best_xsub = None

    for s, c in counts.items():
        Esub, order, xsub = best_energy_for_bitstring(s, Qsub)
        if Esub < local_best_Esub:
            local_best_Esub = Esub
            local_best_s = s
            local_best_order = order
            local_best_xsub = xsub

    # Lift pooled solution into full variable space
    x_full = np.zeros(n_full, dtype=int)
    for pos, orig_i in enumerate(pool_idx):
        x_full[orig_i] = int(local_best_xsub[pos])

    Efull = qubo_energy(Q, x_full)

    if float(Efull) < best["qubo_energy_full"]:
        best.update({
            "qubo_energy_full": float(Efull),
            "qubo_energy_sub": float(local_best_Esub),
            "bitstring": str(local_best_s),
            "bit_order": str(local_best_order),
            "counts_top": dict(sorted(counts.items(), key=lambda kv: kv[1], reverse=True)[:20]),
            "results_json_key": key,
        })

BEST_JSON.write_text(json.dumps(best, indent=2))
print("Wrote:", BEST_JSON)
best


Found 4 results.json file(s) in s3://amazon-braket-us-west-2-581610642254/clinical-trials-data/results/sv1_tasks/scenario_B/
Wrote: data/results/_08a_sv1_best_scenario_B.json


{'source': 's3_results_json',
 'qubo_energy_full': -990.0,
 'qubo_energy_sub': -990.0,
 'bitstring': '0100010010110100000101000001001000',
 'bit_order': 'LR',
 'counts_top': {'0000000000000000000000000000000000': 33,
  '0000100000000000000000000000000000': 3,
  '0000000000000000000010000000000000': 2,
  '0000001000000000000000000000000000': 2,
  '0000000000000000100000000000000000': 2,
  '0000000010000000000000000000000000': 2,
  '0000000000000000010000000000000000': 2,
  '1100111101110111111111011101001111': 1,
  '0111110001011110011111111111001110': 1,
  '0011000010110110001100001011110011': 1,
  '1011110111011011100111011010111111': 1,
  '1000000000000000000000000000000000': 1,
  '0111010001111110000011111011001011': 1,
  '1011111111111001111111111111111011': 1,
  '0001000010000000000000100001000000': 1,
  '0111100111011111111011111101010111': 1,
  '1111110100111111101001111111111111': 1,
  '0101011111111101110110111110100101': 1,
  '1111010110111011111000100111101101': 1,
  '111111

In [9]:
# ============================================================
# Cell 5 — Decode SV1 best → selected trials + export artifacts (Scenario B)
# ============================================================

import json
import numpy as np
import pandas as pd
from pathlib import Path

RESULTS_DIR = Path("data/results")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

BEST_JSON = RESULTS_DIR / "_08a_sv1_best_scenario_B.json"
if not BEST_JSON.exists():
    raise FileNotFoundError(f"Missing: {BEST_JSON} (run Cell 4b2 first).")

PATH_QUBO = Path("data/qubo_scenarios/scenario_B_qubo.json")
if not PATH_QUBO.exists():
    raise FileNotFoundError(f"Missing: {PATH_QUBO} (run 03b first).")

# Optional: scenario B candidates/metadata (used to enrich exported selected trials)
candidate_paths = [
    Path("data/processed/scenario_B_candidates.csv"),
    Path("data/processed/scenario_B_candidates_scored.csv"),
    Path("data/results/scenario_B_candidates.csv"),
    Path("data/results/scenario_B_candidates_scored.csv"),
]
PATH_CANDIDATES = next((p for p in candidate_paths if p.exists()), None)

payload = json.loads(PATH_QUBO.read_text())
Q = np.array(payload["Q"], dtype=float)
n_full = int(Q.shape[0])

nct_ids = payload.get("nct_ids", None)
if not nct_ids or len(nct_ids) != n_full:
    raise ValueError("scenario_B_qubo.json must include nct_ids aligned to Q dimension.")

best = json.loads(BEST_JSON.read_text())

bitstring = str(best["bitstring"]).replace(" ", "")
pool_idx = list(best["pool_idx"])
n_pool = int(best["n_pool"])

if len(pool_idx) != n_pool:
    raise ValueError(f"pool_idx length {len(pool_idx)} != n_pool {n_pool}")

# --- Parse bitstring into pooled x (try LR vs RL and take whichever matches best['bit_order']) ---
def bitstring_to_bits_lr(s, n):
    s = str(s)
    if len(s) != n:
        s = s[:n].ljust(n, "0")
    return np.array([1 if ch == "1" else 0 for ch in s], dtype=int)

x_pool_lr = bitstring_to_bits_lr(bitstring, n_pool)
x_pool_rl = x_pool_lr[::-1].copy()

if best.get("bit_order", "LR") == "RL":
    x_pool = x_pool_rl
else:
    x_pool = x_pool_lr

# --- Lift pooled selection to full 60-length x ---
x_full = np.zeros(n_full, dtype=int)
for pos, orig_i in enumerate(pool_idx):
    x_full[int(orig_i)] = int(x_pool[pos])

selected_full_idx = np.where(x_full == 1)[0].tolist()
selected_nct_ids = [nct_ids[i] for i in selected_full_idx]

print("Scenario B SV1 best:")
print("  n_full:", n_full, "| n_pool:", n_pool)
print("  selected_n (full):", len(selected_full_idx))
print("  qubo_energy_full:", best.get("qubo_energy_full"))
print("  results_json_key:", best.get("results_json_key"))

# --- Build a clean selected-trials table ---
df_sel = pd.DataFrame({
    "nct_id": selected_nct_ids,
    "full_index": selected_full_idx,
})

# Optionally enrich with candidate metadata if available
if PATH_CANDIDATES is not None:
    cand = pd.read_csv(PATH_CANDIDATES)
    if "nct_id" in cand.columns:
        # Keep only useful columns if present
        keep_cols = [c for c in [
            "nct_id", "brief_title", "overall_status", "phase",
            "lead_sponsor", "lead_sponsor_norm", "region_label",
            "estimated_trial_cost", "enrollment_feasibility_score",
            "benefit_score"
        ] if c in cand.columns]
        df_sel = df_sel.merge(cand[keep_cols], on="nct_id", how="left")
        print("Enriched selected trials from:", PATH_CANDIDATES)
    else:
        print("Candidates file found but missing nct_id; skipping enrichment:", PATH_CANDIDATES)

# --- Export artifacts ---
OUT_SELECTED = RESULTS_DIR / "08a_sv1_scenario_B_selected_trials.csv"
OUT_SUMMARY  = RESULTS_DIR / "08a_sv1_scenario_B_summary.csv"
OUT_COUNTS   = RESULTS_DIR / "08a_sv1_scenario_B_counts_top.csv"

df_sel.to_csv(OUT_SELECTED, index=False)

summary = {
    "method": "SV1_QAOA_POOLSUB",
    "scenario": "B",
    "n_full": n_full,
    "n_pool": n_pool,
    "selected_n_full": int(len(selected_full_idx)),
    "qubo_energy_full": float(best.get("qubo_energy_full")),
    "qubo_energy_sub": float(best.get("qubo_energy_sub")),
    "pool_strategy": best.get("pool_strategy"),
    "results_json_key": best.get("results_json_key"),
    "braket_task_bucket": best.get("braket_task_bucket"),
    "braket_task_prefix": best.get("braket_task_prefix"),
}
pd.DataFrame([summary]).to_csv(OUT_SUMMARY, index=False)

counts_top = best.get("counts_top", {}) or {}
pd.DataFrame([{"bitstring": k, "count": v} for k, v in counts_top.items()]).to_csv(OUT_COUNTS, index=False)

print("Wrote:")
print("  -", OUT_SELECTED)
print("  -", OUT_SUMMARY)
print("  -", OUT_COUNTS)

display(df_sel.head(12))
display(pd.DataFrame([summary]))


Scenario B SV1 best:
  n_full: 60 | n_pool: 34
  selected_n (full): 10
  qubo_energy_full: -990.0
  results_json_key: clinical-trials-data/results/sv1_tasks/scenario_B/78808096-278b-4669-b656-361c3d963aec/results.json
Enriched selected trials from: data/processed/scenario_B_candidates.csv
Wrote:
  - data/results/08a_sv1_scenario_B_selected_trials.csv
  - data/results/08a_sv1_scenario_B_summary.csv
  - data/results/08a_sv1_scenario_B_counts_top.csv


,nct_id,full_index,brief_title,overall_status,phase,lead_sponsor
0,NCT06749899,28,QL1706 (PD-1/CTLA-4 Bi-specific Antibody) and ...,RECRUITING,PHASE3,Sun Yat-sen University
1,NCT06744504,31,Standard-dose vs Intermediate-dose Cytarabine ...,RECRUITING,PHASE3,Institute of Hematology & Blood Diseases Hospi...
2,NCT06742723,32,A Phase III Renal Outcomes and Cardiovascular ...,RECRUITING,PHASE3,AstraZeneca
3,NCT06739122,36,A Study of Dulaglutide (LY2189265) 3.0 mg and ...,RECRUITING,PHASE3,Eli Lilly and Company
4,NCT05611801,39,A Clinical Trial of Study Medicine (Marstacima...,RECRUITING,PHASE3,Pfizer
5,NCT05623020,41,A Study to Learn About the Effects of the Comb...,RECRUITING,PHASE3,Pfizer
6,NCT05624450,42,Efficacy and Safety of Tozorakimab in Patients...,RECRUITING,PHASE3,AstraZeneca
7,NCT06732401,45,Testing the Addition of AZD6738 (Ceralasertib)...,RECRUITING,PHASE3,National Cancer Institute (NCI)
8,NCT05577988,50,Assessment of an Early De-Escalation to a Low-...,RECRUITING,PHASE3,Assistance Publique - Hôpitaux de Paris
9,NCT05794971,52,Regorafenib Combined With Irinotecan Drug-Elut...,RECRUITING,PHASE3,Sun Yat-sen University


,method,scenario,n_full,n_pool,selected_n_full,qubo_energy_full,qubo_energy_sub,pool_strategy,results_json_key,braket_task_bucket,braket_task_prefix
0,SV1_QAOA_POOLSUB,B,60,34,10,-990.0,-990.0,best_diag,clinical-trials-data/results/sv1_tasks/scenari...,amazon-braket-us-west-2-581610642254,clinical-trials-data/results/sv1_tasks/scenario_B


### What Cell 5 Just Did

- Loaded the recovered SV1 “best” record (`_08a_sv1_best_scenario_B.json`) and decoded the pooled bitstring into a pooled selection vector.
- Lifted that pooled selection into the full Scenario B 60-variable space, producing a list of selected trials (`nct_id`s).
- Optionally enriched the selection with Scenario B candidate metadata (title/status/phase/sponsor/region/cost/feasibility/benefit) when the candidates CSV is available.
- Exported portfolio-ready artifacts:
  - `data/results/08a_sv1_scenario_B_selected_trials.csv`
  - `data/results/08a_sv1_scenario_B_summary.csv`
  - `data/results/08a_sv1_scenario_B_counts_top.csv`


## Summary (Scenario B — AWS Braket SV1 QAOA)

This notebook executed a Scenario B QAOA run on AWS Braket’s SV1 simulator using an SV1-safe pooling strategy (reducing the full Scenario B QUBO from 60 binary decision variables down to a 34-variable subproblem to respect the device qubit limit). Because one local job stalled after the Braket tasks completed, we recovered deterministically by reading the completed Braket `results.json` outputs directly from the `amazon-braket-us-west-2-581610642254` bucket under `clinical-trials-data/results/sv1_tasks/scenario_B/` and reconstructing measurement counts from the `measurements` field. We then selected the best-scoring bitstring by evaluating QUBO energy (checking both bit orders for robustness), lifted the pooled solution back into the full 60-variable space, and exported portfolio-ready artifacts.

Key outputs written:
- `data/results/_08a_sv1_best_scenario_B.json` — recovered “best” SV1 record (bitstring, pool indices, energies, and top counts)
- `data/results/08a_sv1_scenario_B_selected_trials.csv` — decoded selected Scenario B trials (NCT IDs) with optional metadata enrichment
- `data/results/08a_sv1_scenario_B_summary.csv` — run summary (dimensions, energies, pooling strategy, and Braket output location)
- `data/results/08a_sv1_scenario_B_counts_top.csv` — top observed bitstrings and counts from the winning SV1 task

Result highlight:
- Best recovered full-QUBO energy: `-990.0` (from Braket SV1 task outputs)

Next steps:
- Upload the exported artifacts to your results S3 location (if desired) and commit/push the notebook + artifacts to GitHub.
- For a “fuller demo,” expand the (gamma, beta) grid and/or shots count while keeping the SV1 pooling constraint (≤34 variables) to stay within SV1 limits.
